## Carga de Datos

In [5]:
from pathlib import Path

import polars as pl
from loguru import logger

from sklearn.model_selection import train_test_split

SEED = 42

In [6]:
DATA_DIR = Path("../data/processed")

def load_processed_data(folder: str, format: str = "parquet"):

    logger.info(f'Cargando datos desde {folder} con formato {format}...')
    return pl.scan_parquet(f"{folder}/*.{format}").collect()

In [8]:
cleaned_dataset = load_processed_data(DATA_DIR, format="parquet")
cleaned_dataset.head(5).show()

2026-05-14 14:19:53.332 | INFO     | __main__:load_processed_data:5 - Cargando datos desde ../data/processed con formato parquet...


cole_sede_principal,cole_caracter,fami_cuartoshogar,fami_educacionpadre,estu_tipodocumento,estu_fechanacimiento,fami_tienecomputador,estu_genero,fami_tienelavadora,fami_personashogar,estu_depto_presentacion,estu_depto_reside,cole_naturaleza,cole_calendario,estu_mcpio_reside,periodo,cole_genero,fami_estratovivienda,cole_area_ubicacion,cole_bilingue,cole_mcpio_ubicacion,estu_mcpio_presentacion,fami_educacionmadre,fami_tieneautomovil,cole_jornada,punt_global,fami_tieneinternet,cole_depto_ubicacion
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""N""","""TÉCNICO/ACADÉMICO""","""Dos""","""Primaria completa""","""TI""","""15/02/2003""","""Si""","""F""","""Si""","""1 a 2""","""HUILA""","""HUILA""","""OFICIAL""","""A""","""AIPE""","""20194""","""MIXTO""","""Estrato 2""","""RURAL""","""N""","""AIPE""","""AIPE""","""Postgrado""","""No""","""COMPLETA""","""339""","""Si""","""HUILA"""
"""S""","""TÉCNICO/ACADÉMICO""","""Dos""","""Primaria incompleta""","""TI""","""08/03/2002""","""No""","""F""","""No""","""5 a 6""","""HUILA""","""HUILA""","""OFICIAL""","""A""","""LA PLATA""","""20194""","""MIXTO""","""Estrato 1""","""URBANO""","""N""","""LA PLATA""","""LA PLATA""","""Primaria incompleta""","""No""","""COMPLETA""","""199""","""No""","""HUILA"""
"""S""","""TÉCNICO/ACADÉMICO""","""Cuatro""","""Secundaria (Bachillerato) comp…","""TI""","""27/07/2000""","""Si""","""M""","""Si""","""Cinco""","""VALLE""","""VALLE""","""OFICIAL""","""A""","""CALI""","""20162""","""MIXTO""","""Estrato 5""","""URBANO""","""N""","""CALI""","""CALI""","""Secundaria (Bachillerato) inco…","""Si""","""MAÑANA""","""272""","""Si""","""VALLE"""
"""S""","""TÉCNICO/ACADÉMICO""","""Cinco""","""Primaria incompleta""","""TI""","""07/12/1999""","""No""","""F""","""Si""","""5 a 6""","""ANTIOQUIA""","""ANTIOQUIA""","""NO OFICIAL""","""A""","""MEDELLÍN""","""20172""","""MIXTO""","""Estrato 2""","""URBANO""","""N""","""MEDELLIN""","""ITAGÜÍ""","""Secundaria (Bachillerato) inco…","""No""","""SABATINA""","""253""","""Si""","""ANTIOQUIA"""
"""S""","""TÉCNICO/ACADÉMICO""","""Dos""","""Primaria incompleta""","""CC""","""15/09/1995""","""No""","""M""","""No""","""Cuatro""","""TOLIMA""","""TOLIMA""","""OFICIAL""","""A""","""SAN LUIS""","""20142""","""MIXTO""","""Estrato 1""","""RURAL""","""N""","""SAN LUIS""","""GUAMO""","""Primaria incompleta""","""No""","""MAÑANA""","""212""","""No""","""TOLIMA"""


## Split estratificado por Período

In [9]:
print(cleaned_dataset['periodo'].value_counts().sort(by='periodo').head(10))

shape: (10, 2)
┌─────────┬────────┐
│ periodo ┆ count  │
│ ---     ┆ ---    │
│ str     ┆ u32    │
╞═════════╪════════╡
│ 20142   ┆ 544535 │
│ 20151   ┆ 25947  │
│ 20152   ┆ 542450 │
│ 20161   ┆ 13064  │
│ 20162   ┆ 548206 │
│ 20171   ┆ 13018  │
│ 20172   ┆ 546261 │
│ 20181   ┆ 19798  │
│ 20191   ┆ 12540  │
│ 20194   ┆ 546211 │
└─────────┴────────┘


In [10]:
X = cleaned_dataset.drop("punt_global")
y = cleaned_dataset["punt_global"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=cleaned_dataset["periodo"]
)

print(f"Train: {X_train.shape[0]:,} filas | Test: {X_test.shape[0]:,} filas")
print(f"Proporcion test: {X_test.shape[0] / cleaned_dataset.shape[0] * 100:.1f}%")

Train: 2,716,667 filas | Test: 679,167 filas
Proporcion test: 20.0%


# Pipeline

In [11]:
import numpy as np
import pandas as pd

import optuna
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Ridge, Lasso, LinearRegression

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [12]:
def regression_report(y_true: pd.Series, y_pred: np.ndarray, mod_tag: str = "default") -> dict:

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {
        "model-experiment": mod_tag,
        "r2": r2,
        "rmse": rmse,
        "mae": mae,
    }


def evaluate_model(
    estimator,
    X_train,
    y_train,
    X_test,
    y_test,
    mod_tag: str
) -> dict:

    estimator.fit(X_train, y_train)
    predictions = estimator.predict(X_test)
    
    report = regression_report(y_test, predictions, mod_tag)
    logger.info(f"Evaluación del modelo {mod_tag}: {report}")

    return report


def deterministic_processor(
    X: pl.DataFrame,
    mapeo_personas: dict,
    mapeo_cuartos: dict
) -> pl.DataFrame:

    X_calculated = (
        X.clone()
        .with_columns(
        (
            pl.col("periodo").str.slice(0, 4).cast(pl.Int32) - 
            pl.col("estu_fechanacimiento").str.slice(6, 4).cast(pl.Int32)
        )
        .alias("edad_estudiante_aprox")
        )
        .with_columns(
            pl.col('edad_estudiante_aprox')
            .clip(15, 60)
            .alias('edad_estudiante_aprox')
        )
        .with_columns(
            pl.col("fami_personashogar")
            .replace(mapeo_personas)
            .cast(pl.Int32, strict=False)
            .alias("fami_personashogar_num"),

            pl.col("fami_cuartoshogar")
            .replace(mapeo_cuartos)
            .cast(pl.Int32, strict=False)
            .alias("fami_cuartos_num")
        )
        .drop([
            "fami_personashogar",
            "fami_cuartoshogar",
            "estu_fechanacimiento"
        ])
        .with_columns(
            ((pl.col("fami_tieneinternet") == "Si").cast(pl.Int8) + 
            (pl.col("fami_tienecomputador") == "Si").cast(pl.Int8))
            .alias("capital_tecnologico")
        )
    )

    return X_calculated

In [13]:
mapeo_personas = {
    "Uno": 1,
    "Dos": 2,
    "Tres": 3,
    "Cuatro": 4,
    "Cinco": 5,
    "Seis": 6,
    "Siete": 7,
    "Ocho": 8,
    "Nueve": 9,
    "Diez": 10,
    "Once": 11,
    "Doce o más": 12,
    "1 a 2": 1,
    "3 a 4": 3,
    "5 a 6": 5,
    "7 a 8": 7,
    "9 o más": 9
}


mapeo_cuartos = {
    "Uno": 1,
    "Dos": 2,
    "Tres": 3,
    "Cuatro": 4,
    "Cinco": 5,
    "Seis": 6,
    "Seis o mas": 6,
    "Siete": 7,
    "Ocho": 8,
    "Nueve": 9,
    "Diez o más": 10,
}

In [14]:
X_train = deterministic_processor(X_train.clone(), mapeo_personas, mapeo_cuartos)
X_test = deterministic_processor(X_test.clone(), mapeo_personas, mapeo_cuartos)

X_train_pd = X_train.to_pandas()
X_test_pd = X_test.to_pandas()

In [15]:
# Definir orden de los estratos:
orden_estrato = [
    'Sin Estrato',
    'Estrato 1',
    'Estrato 2',
    'Estrato 3',
    'Estrato 4',
    'Estrato 5',
    'Estrato 6'
]

# Definir orden de la educación:
orden_educacion = [
    'Ninguno',
    'No sabe',
    "No Aplica",
    'Primaria incompleta',
    'Primaria completa',
    'Secundaria (Bachillerato) incompleta',
    'Secundaria (Bachillerato) completa',
    'Técnica o tecnológica incompleta',
    'Técnica o tecnológica completa',
    'Educación profesional incompleta',
    'Educación profesional completa',
    'Postgrado',
]

# Clasificación de las variables:
ordinal_cols = [
    'fami_estratovivienda',
    'fami_educacionmadre',
    'fami_educacionpadre'
]

binary_cols = [
    'fami_tieneinternet',
    'fami_tienecomputador',
    'fami_tieneautomovil',
    'fami_tienelavadora',
    'cole_bilingue',
    'cole_sede_principal',
    'estu_genero',
]

nominal_low_cols = [
    'cole_naturaleza',
    'cole_area_ubicacion',
    'cole_calendario',
    'cole_jornada',
    'cole_genero',
    'cole_caracter',
    'estu_tipodocumento',
    'estu_depto_reside',
    'estu_depto_presentacion',
    'cole_depto_ubicacion',
    'periodo',
]

nominal_high_cols = [
    'estu_mcpio_reside',
    'cole_mcpio_ubicacion',
    'estu_mcpio_presentacion',
]

numeric_cols = [
    'edad_estudiante_aprox',
    'fami_personashogar_num',
    'fami_cuartos_num',
    "capital_tecnologico",
]

# Sub-pipelines
preprocessor = ColumnTransformer([
    ("ordinal", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(
            categories=[orden_estrato, orden_educacion, orden_educacion],
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )),
    ]), ordinal_cols),

    ("binary", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ]), binary_cols),

    ("nominal_low", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), nominal_low_cols),

    ("nominal_high", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ]), nominal_high_cols),

    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ]), numeric_cols),
], remainder="drop")

In [11]:
linear_report = []

linear_model_specs = [
    ("LinearRegression", LinearRegression(), True),
    ("Ridge", Ridge(alpha=50), True),
    ("Lasso", Lasso(alpha=50), True),
]

for mod_tag, regressor, scale_flag in linear_model_specs:

    linear_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("scaler", StandardScaler() if scale_flag else "passthrough"),
        ("model", regressor),
    ])

    reg_report = evaluate_model(
        estimator=linear_pipeline,
        X_train=X_train_pd,
        y_train=y_train,
        X_test=X_test_pd,
        y_test=y_test,
        mod_tag=mod_tag
    )

    linear_report.append(reg_report)


linear_results_df = pd.DataFrame(linear_report).sort_values("rmse")
display(linear_results_df)

2026-05-14 08:32:02.480 | INFO     | __main__:evaluate_model:28 - Evaluación del modelo LinearRegression: {'model-experiment': 'LinearRegression', 'r2': 0.35355604270441265, 'rmse': np.float64(40.05840636568485), 'mae': 32.077695365906095}
2026-05-14 08:32:31.693 | INFO     | __main__:evaluate_model:28 - Evaluación del modelo Ridge: {'model-experiment': 'Ridge', 'r2': 0.35355593972416244, 'rmse': np.float64(40.05840955639032), 'mae': 32.07770728581614}
2026-05-14 08:32:58.350 | INFO     | __main__:evaluate_model:28 - Evaluación del modelo Lasso: {'model-experiment': 'Lasso', 'r2': -2.34761432649222e-06, 'rmse': np.float64(49.822869597799745), 'mae': 40.39893859721917}


,model-experiment,r2,rmse,mae
0,LinearRegression,0.353556,40.058406,32.077695
1,Ridge,0.353556,40.058410,32.077707
2,Lasso,-0.000002,49.822870,40.398939


In [ ]:
tree_report = []

tree_model_specs = [
    ("Tree", DecisionTreeRegressor(max_depth=8, random_state=SEED), False),
    ("RandomForest", RandomForestRegressor(n_estimators=400, max_depth=12, random_state=SEED), False),
    ("GradientBoosting", GradientBoostingRegressor(n_estimators=400, max_depth=6, random_state=SEED), False),
    ("LightGBM", LGBMRegressor(n_estimators=400, max_depth=6, random_state=SEED), False),
    ("XGBoost", XGBRegressor(n_estimators=400, max_depth=6, random_state=SEED), False),
    ("CatBoost", CatBoostRegressor(n_estimators=400, max_depth=6, random_state=SEED, verbose=0), False)
]

for mod_tag, regressor, scale_flag in tree_model_specs:

    tree_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", regressor),
    ])

    reg_report = evaluate_model(
        estimator=tree_pipeline,
        X_train=X_train_pd,
        y_train=y_train,
        X_test=X_test_pd,
        y_test=y_test,
        mod_tag=mod_tag
    )

    tree_report.append(reg_report)

tree_results_df = pd.DataFrame(tree_report).sort_values("rmse")
display(tree_results_df)

2026-05-14 08:33:48.628 | INFO     | __main__:evaluate_model:28 - Evaluación del modelo Tree: {'model-experiment': 'Tree', 'r2': 0.3406534145174649, 'rmse': np.float64(40.45620194587893), 'mae': 32.43469957173675}


In [1]:
from optuna import Trial, create_study, logging

logging.set_verbosity(logging.WARNING)  # menos ruido

N_TRIALS_LINEAR = 8    # Ridge/Lasso: solo 1 param, converge rápido
N_TRIALS_TREE = 15     # Árboles: más params, necesitan más exploración
CV_FOLDS = 3           # Tuning inicial
CV_FOLDS_FINAL = 5     # Evaluación final del ganador por modelo
PARALLEL_TRIALS = 2    # Trials en paralelo para árboles
CORES_PER_MODEL = 5    # 10 cores / 2 trials paralelos

/Users/santiago.higuitau/Documents/Machine_Learning/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def build_model_grid(trial: Trial, model_selected: str) -> dict:

    if model_selected == "ridge":
        return {
            "model_type": "linear",
            "params_grid": {
                "alpha": trial.suggest_float("alpha", 1e-3, 1000.0, log=True),
            },
        }

    elif model_selected == "lasso":
        return {
            "model_type": "linear",
            "params_grid": {
                "alpha": trial.suggest_float("alpha", 1, 500.0, log=True),
                "max_iter": 2000,
            },
        }

    elif model_selected == "randomForest":
        return {
            "model_type": "tree",
            "params_grid": {
                "n_estimators": trial.suggest_int("n_estimators", 100, 500),
                "max_depth": trial.suggest_int("max_depth", 6, 20),
                "min_samples_split": trial.suggest_int("min_samples_split", 5, 50),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 30),
                "max_features": trial.suggest_float("max_features", 0.3, 1.0),
            },
        }

    elif model_selected == "lightGBM":
        return {
            "model_type": "tree",
            "params_grid": {
                "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
                "max_depth": trial.suggest_int("max_depth", 4, 12),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "num_leaves": trial.suggest_int("num_leaves", 20, 150),
                "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
                "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            },
        }

    elif model_selected == "xgboost":
        return {
            "model_type": "tree",
            "params_grid": {
                "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
                "max_depth": trial.suggest_int("max_depth", 4, 12),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
                "min_child_weight": trial.suggest_int("min_child_weight", 1, 50),
                "gamma": trial.suggest_float("gamma", 1e-8, 5.0, log=True),
            },
        }

    elif model_selected == "catboost":
        return {
            "model_type": "tree",
            "params_grid": {
                "iterations": trial.suggest_int("iterations", 200, 1000),
                "depth": trial.suggest_int("depth", 4, 10),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
                "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
                "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            },
        }

    return {}

In [3]:
def objective_model(trial, model_selected: str, random_seed: int):

    config = build_model_grid(trial, model_selected)
    params = config.get("params_grid", {})
    model_type = config.get("model_type", None)

    assert params, f"Modelo {model_selected} sin parámetros en la configuración."
    assert model_type is not None, f"Modelo {model_selected} no encontrado en la configuración."


    if model_type == 'linear':

        model = Ridge(**params) if model_selected == 'ridge' else Lasso(**params)

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("scaler", StandardScaler()),
            ("model", model),
        ])


    elif model_type == 'tree':

        if model_selected == 'randomForest':
            model = RandomForestRegressor(**params, random_state=random_seed, n_jobs=CORES_PER_MODEL)

        elif model_selected == 'lightGBM':
            model = LGBMRegressor(**params, random_state=random_seed, verbose=-1, n_jobs=CORES_PER_MODEL)

        elif model_selected == 'xgboost':
            model = XGBRegressor(**params, random_state=random_seed, verbosity=0, tree_method="hist", nthread=CORES_PER_MODEL)

        else:
            model = CatBoostRegressor(**params, random_state=random_seed, verbose=0, thread_count=CORES_PER_MODEL)

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model),
        ])

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=CV_FOLDS,
        scoring="neg_root_mean_squared_error",
        n_jobs=1
    )

    return -scores.mean()

In [4]:
studies = dict()
models_to_tune = [
    'ridge',
    'lasso',
    'randomForest',
    'lightGBM',
    'xgboost',
    'catboost'
]

for model_name in models_to_tune:
    logger.info(f'Iniciando tuning usando Optuna para el modelo: {model_name}')

    # Trials y paralelismo según tipo de modelo
    is_linear = model_name in ['ridge', 'lasso']
    n_trials = N_TRIALS_LINEAR if is_linear else N_TRIALS_TREE
    n_jobs_optuna = 1 if is_linear else PARALLEL_TRIALS

    study = create_study(direction="minimize", study_name=f'Experimento usando: {model_name}')
    study.optimize(
        lambda trial, m=model_name, r=SEED: objective_model(trial, model_selected=m, random_seed=r),
        n_trials=n_trials,
        n_jobs=n_jobs_optuna,
        show_progress_bar=True
    )

    studies[model_name] = study

    logger.info(f'Mejor RMSE CV: {study.best_value:.4f}')
    logger.info(f'Mejores params: {study.best_params}')

NameError: name 'logger' is not defined